# 00 — Data Exploration: Parsertime OW2 Dataset

This notebook profiles the [luxdotdev/dataset](https://github.com/luxdotdev/dataset) — approximately **1,900 anonymized competitive Overwatch 2 matches** captured via Parsertime/ScrimTime workshop codes.

**Goals:**
- Profile all 24 tables: row counts, schemas, missing values
- Understand the entity hierarchy (Scrim → Match → Round → Events)
- Assess data quality: orphaned records, timestamp consistency
- Establish a foundation for all subsequent analysis notebooks

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_all, load_csv, ALL_TABLES
from src.visualization import setup_style, OW_COLORS, OW_PALETTE

setup_style()
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 40)

## 1. Load All Tables

In [ ]:
tables = load_all()

# Summary: row counts and column counts
summary = pd.DataFrame([
    {"Table": name, "Rows": len(df), "Columns": len(df.columns), "Memory (MB)": df.memory_usage(deep=True).sum() / 1e6}
    for name, df in tables.items()
]).sort_values("Rows", ascending=False)

print(f"Total tables: {len(tables)}")
print(f"Total rows:   {summary['Rows'].sum():,}")
print(f"Total memory: {summary['Memory (MB)'].sum():.1f} MB")
print()
summary

## 2. Schema Profiles

Let's look at each table's dtypes, missing values, and unique counts for key columns.

In [ ]:
for name in ['Kill', 'PlayerStat', 'MatchStart', 'MatchEnd', 'Scrim', 'RoundStart', 'RoundEnd']:
    df = tables[name]
    print(f"\n{'='*60}")
    print(f"TABLE: {name} ({len(df):,} rows, {len(df.columns)} cols)")
    print(f"{'='*60}")
    
    info_df = pd.DataFrame({
        'dtype': df.dtypes,
        'non_null': df.count(),
        'null_count': df.isnull().sum(),
        'null_pct': (df.isnull().sum() / len(df) * 100).round(1),
        'unique': df.nunique(),
    })
    print(info_df.to_string())
    print()

## 3. Entity Hierarchy

The data follows this hierarchy:
- **Scrim**: A scrim session (multiple matches)
- **Match**: A single map played (linked via `MapDataId` and `scrimId`)
- **Round**: Rounds within a match
- **Events**: Kill, Ult, Assist, etc. events within rounds

In [ ]:
scrims = tables['Scrim']
match_start = tables['MatchStart']
match_end = tables['MatchEnd']
round_start = tables['RoundStart']
round_end = tables['RoundEnd']
kills = tables['Kill']

print(f"Scrims:         {len(scrims):,}")
print(f"Matches:        {match_start['MapDataId'].nunique():,}")
print(f"Rounds:         {round_start.shape[0]:,}")
print(f"Kills:          {len(kills):,}")
print(f"Unique teams:   {kills['attacker_team'].nunique():,}")
print(f"Unique players: {kills['attacker_name'].nunique():,}")
print(f"Unique heroes:  {kills['attacker_hero'].nunique():,}")
print()

# Date range (format='mixed' handles varying precision in timestamps)
scrims['date_parsed'] = pd.to_datetime(scrims['date'], format='mixed')
print(f"Date range: {scrims['date_parsed'].min().date()} to {scrims['date_parsed'].max().date()}")
print(f"Span: {(scrims['date_parsed'].max() - scrims['date_parsed'].min()).days} days")

In [ ]:
# Matches per scrim distribution
matches_per_scrim = match_start.groupby('scrimId').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(matches_per_scrim, bins=range(1, matches_per_scrim.max() + 2),
             color=OW_COLORS['orange'], edgecolor=OW_COLORS['dark_blue'], alpha=0.9)
axes[0].set_xlabel('Matches per Scrim')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Matches per Scrim Session')

# Map type distribution
map_type_counts = match_start['map_type'].value_counts()
axes[1].barh(map_type_counts.index, map_type_counts.values,
             color=OW_PALETTE[:len(map_type_counts)])
axes[1].set_xlabel('Number of Matches')
axes[1].set_title('Matches by Map Type')

plt.tight_layout()
plt.show()

## 4. Map Distribution

In [ ]:
map_counts = match_start['map_name'].value_counts()

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(map_counts.index[::-1], map_counts.values[::-1],
               color=OW_COLORS['blue'], alpha=0.85)
ax.set_xlabel('Number of Matches')
ax.set_title('Matches by Map')

# Add map type annotation
map_types = match_start.drop_duplicates('map_name').set_index('map_name')['map_type']
for bar, map_name in zip(bars, map_counts.index[::-1]):
    mtype = map_types.get(map_name, '?')
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'  {mtype}', va='center', fontsize=9, color=OW_COLORS['light_gray'])

plt.tight_layout()
plt.show()

## 5. Hero Distribution

In [ ]:
from src.preprocessing import HERO_ROLES
from src.visualization import ROLE_COLORS
from matplotlib.patches import Patch

# Hero appearances from Kill table (both attacker and victim)
hero_kills = kills['attacker_hero'].value_counts()
hero_deaths = kills['victim_hero'].value_counts()

# All unique heroes seen
all_heroes = set(kills['attacker_hero'].unique()) | set(kills['victim_hero'].unique())
print(f"Unique heroes in dataset: {len(all_heroes)}")

# Check for heroes not in our role mapping
unknown = all_heroes - set(HERO_ROLES.keys())
if unknown:
    print(f"Heroes not in role mapping: {unknown}")
else:
    print("All heroes mapped to roles successfully!")

# Top killers
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top_n = 20
colors_kill = [ROLE_COLORS.get(HERO_ROLES.get(h, 'Unknown'), OW_COLORS['light_gray']) 
               for h in hero_kills.head(top_n).index]
axes[0].barh(hero_kills.head(top_n).index[::-1], hero_kills.head(top_n).values[::-1],
             color=colors_kill[::-1])
axes[0].set_title('Top Heroes by Kills Secured')
axes[0].set_xlabel('Total Kills')

colors_death = [ROLE_COLORS.get(HERO_ROLES.get(h, 'Unknown'), OW_COLORS['light_gray']) 
                for h in hero_deaths.head(top_n).index]
axes[1].barh(hero_deaths.head(top_n).index[::-1], hero_deaths.head(top_n).values[::-1],
             color=colors_death[::-1])
axes[1].set_title('Top Heroes by Deaths')
axes[1].set_xlabel('Total Deaths')

# Add legend for roles
legend_elements = [Patch(facecolor=c, label=r) for r, c in ROLE_COLORS.items()]
axes[0].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## 6. Data Quality Checks

In [ ]:
# Check: Do all kill MapDataIds have a corresponding MatchStart?
match_ids = set(match_start['MapDataId'].unique())
kill_map_ids = set(kills['MapDataId'].unique())
orphaned_kills = kill_map_ids - match_ids
print(f"Kill MapDataIds with no MatchStart: {len(orphaned_kills)}")

# Check: Do all MatchStarts have a MatchEnd?
end_ids = set(match_end['MapDataId'].unique())
missing_ends = match_ids - end_ids
print(f"Matches without MatchEnd: {len(missing_ends)}")

# Check: Scrim IDs referenced by matches
scrim_ids = set(scrims['id'].unique())
match_scrim_ids = set(match_start['scrimId'].unique())
orphaned_matches = match_scrim_ids - scrim_ids
print(f"Match scrimIds with no Scrim record: {len(orphaned_matches)}")

# Check: match_time consistency (should be non-negative and reasonable)
print(f"\nKill match_time range: {kills['match_time'].min():.1f}s to {kills['match_time'].max():.1f}s")
negative_times = (kills['match_time'] < 0).sum()
print(f"Negative match_times in kills: {negative_times}")

# Self-kills (cast to str to avoid categorical comparison issues)
self_kills = kills[kills['attacker_name'].astype(str) == kills['victim_name'].astype(str)]
print(f"\nSelf-kills (environmental/self-damage): {len(self_kills):,}")

# Team kills (same team)
team_kills = kills[kills['attacker_team'].astype(str) == kills['victim_team'].astype(str)]
print(f"Same-team kills (should be env/self only): {len(team_kills):,}")

## 7. PlayerStat Table Deep Dive

The PlayerStat table is the richest source of per-player metrics. Let's validate it.

In [ ]:
ps = tables['PlayerStat']

print(f"Total PlayerStat rows: {len(ps):,}")
print(f"Unique players: {ps['player_name'].nunique()}")
print(f"Unique heroes: {ps['player_hero'].nunique()}")
print(f"Unique matches (MapDataId): {ps['MapDataId'].nunique()}")
print(f"Unique rounds: {ps.groupby(['MapDataId', 'round_number']).ngroups}")
print()

# Key stat distributions
stat_cols = ['eliminations', 'final_blows', 'deaths', 'hero_damage_dealt', 
             'healing_dealt', 'hero_time_played']
print(ps[stat_cols].describe().round(1))

In [ ]:
# Cross-validate: total kills in Kill.csv vs sum of final_blows in PlayerStat
total_kills_kill_table = len(kills)
total_fb_playerstat = ps['final_blows'].sum()

print(f"Total events in Kill.csv:       {total_kills_kill_table:,}")
print(f"Sum of final_blows (PlayerStat): {total_fb_playerstat:,.0f}")
print(f"Note: PlayerStat accumulates per round, so final_blows sums will differ.")
print(f"      Kill.csv is the ground truth for individual kill events.")

## 8. Event Tables Overview

In [ ]:
# Event table sizes visualization
event_sizes = pd.Series({
    name: len(tables[name]) 
    for name in sorted(tables.keys(), key=lambda x: len(tables[x]), reverse=True)
})

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(event_sizes.index[::-1], event_sizes.values[::-1],
               color=OW_COLORS['teal'], alpha=0.85)
ax.set_xlabel('Number of Rows')
ax.set_title('Dataset Table Sizes')
ax.set_xscale('log')

for bar, val in zip(bars, event_sizes.values[::-1]):
    ax.text(val * 1.1, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, color=OW_COLORS['white'])

plt.tight_layout()
plt.show()

## 9. Summary

### Key Findings

| Metric | Value |
|--------|-------|
| Total tables | 24 |
| Total rows | ~2M |
| Scrims | ~1,000 |
| Matches | ~4,800 |
| Kills | ~373K |
| PlayerStat rows | ~253K |

### Data Quality
- All player/team names are anonymized (hashed)
- `MapDataId` is the primary join key linking events to matches
- `scrimId` links matches to scrim sessions
- Most tables are well-formed with minimal missing data

### Table Relationships
```
Scrim (id) ──1:N──► MatchStart (scrimId, MapDataId)
                          │
                     MapDataId
                          │
                ┌─────────┼──────────┐
                ▼         ▼          ▼
           RoundStart  Kill     PlayerStat
           RoundEnd    Assist   UltimateCharged
           MatchEnd    HeroSwap UltimateStart/End
                       MercyRez DvaRemech
                       Objective tables...
```

### Next Steps
- **Notebook 01**: Fight detection + first death analysis
- **Notebook 02**: Ultimate economy patterns
- **Notebook 03**: Hero composition analysis